In [4]:
import kagglehub
import pandas as pd
import numpy as np
import os

path = kagglehub.dataset_download("chethuhn/network-intrusion-dataset")
print(" Path to dataset files:", path)

files = [
'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv',
'Monday-WorkingHours.pcap_ISCX.csv',
'Friday-WorkingHours-Morning.pcap_ISCX.csv',
'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',
'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',
'Tuesday-WorkingHours.pcap_ISCX.csv',
'Wednesday-workingHours.pcap_ISCX.csv',
'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv'
       
       ]

dfs = []

for file in files:
    df = pd.read_csv(os.path.join(path, file))
    df.columns = df.columns.str.strip()
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
print(data.shape)
print(data['Label'].value_counts())

 Path to dataset files: /Users/antoniogonzalez/.cache/kagglehub/datasets/chethuhn/network-intrusion-dataset/versions/1
(2830743, 79)
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()


data['Label'] = le.fit_transform(data['Label'])

data.replace([np.inf, -np.inf], np.nan, inplace=True)
data.dropna(inplace=True)

X = data.drop('Label', axis=1)
y = data['Label']

In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=100)

X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)
y_train_t = torch.LongTensor(y_train.values)
y_test_t = torch.LongTensor(y_test.values)

class NeuralNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NeuralNet, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.layers(x)
    

input_size = X_train_t.shape[1]
num_classes = len(y.unique())
model = NeuralNet(input_size, num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

dataset = TensorDataset(X_train_t, y_train_t)
loader = DataLoader(dataset, batch_size=512, shuffle=True)

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model.to(device)

for epoch in range(10):
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        output = model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch+1}/10 Loss: {loss.item():.4f}')


    model.eval()
    with torch.no_grad():
        X_test_device = X_test_t.to(device)
        y_pred = model(X_test_device).argmax(dim=1).cpu().numpy()

    print(classification_report(y_test, y_pred))

Epoch 1/10 Loss: 0.0315


/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       0.98      0.99      0.99    454144
           1       0.96      0.31      0.47       402
           2       1.00      0.98      0.99     25672
           3       0.98      0.95      0.96      2093
           4       0.99      0.92      0.95     45883
           5       0.89      0.89      0.89      1119
           6       0.91      0.92      0.91      1147
           7       0.99      0.49      0.66      1619
           8       0.00      0.00      0.00         1
           9       0.00      0.00      0.00         5
          10       0.87      0.96      0.92     31867
          11       0.90      0.99      0.94      1182
          12       0.00      0.00      0.00       319
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00       121

    accuracy                           0.98    565576
   macro avg       0.63      0.56      0.58    565576
weighted avg       0.98   

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0       1.00      0.98      0.99    454144
           1       0.96      0.31      0.47       402
           2       1.00      1.00      1.00     25672
           3       0.97      0.98      0.97      2093
           4       0.92      1.00      0.96     45883
           5       0.90      0.97      0.93      1119
           6       0.96      0.95      0.96      1147
           7       0.97      0.93      0.95      1619
           8       1.00      1.00      1.00         1
           9       1.00      0.20      0.33         5
          10       0.89      0.97      0.93     31867
          11       0.95      0.90      0.92      1182
          12       0.87      0.04      0.08       319
          13       0.00      0.00      0.00         2
          14       0.00      0.00      0.00       121

    accuracy                           0.98    565576
   macro avg       0.82      0.68      0.70    565576
weighted avg       0.98   

...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 9/10 Loss: 0.0176
              precision    recall  f1-score   support

           0       0.99      1.00      0.99    454144
           1       0.95      0.35      0.52       402
           2       1.00      1.00      1.00     25672
           3       0.98      0.98      0.98      2093
           4       0.99      0.96      0.97     45883
           5       0.90      0.98      0.94      1119
           6       0.98      0.96      0.97      1147
           7       0.99      0.97      0.98      1619
           8       1.00      1.00      1.00         1
           9       0.30      0.60      0.40         5
          10       0.99      0.93      0.96     31867
          11       0.96      0.97      0.96      1182
          12       1.00      0.04      0.08       319
          13       0.00      0.00      0.00         2
          14       1.00      0.02      0.03       121

    accuracy                           0.99    565576
   macro avg       0.87      0.72      0.72    565576
we

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 10/10 Loss: 0.0102
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    454144
           1       0.96      0.32      0.48       402
           2       1.00      1.00      1.00     25672
           3       0.98      0.99      0.98      2093
           4       0.95      1.00      0.97     45883
           5       0.91      0.98      0.94      1119
           6       0.98      0.97      0.97      1147
           7       0.98      0.99      0.99      1619
           8       1.00      1.00      1.00         1
           9       0.50      0.20      0.29         5
          10       0.99      1.00      0.99     31867
          11       0.97      0.97      0.97      1182
          12       1.00      0.09      0.17       319
          13       0.00      0.00      0.00         2
          14       1.00      0.02      0.03       121

    accuracy                           0.99    565576
   macro avg       0.88      0.70      0.72    565576
w

/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/opt/anaconda3/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
